## Projects



In [7]:
import cv2
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

from skimage.feature import hog
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

from tensorflow.keras.utils import to_categorical

In [5]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

In [6]:
dataset_path = "dataset"

images = []
labels = []

label_map = {}
current_label = 0

IMG_SIZE = 128

In [9]:
for person_name in os.listdir(dataset_path):

    person_folder = os.path.join(
        dataset_path,
        person_name
    )

    if not os.path.isdir(person_folder):
        continue

    label_map[current_label] = person_name

    for image_name in os.listdir(person_folder):

        image_path = os.path.join(
            person_folder,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5
        )

        for (x, y, w, h) in faces:

            face = gray[y:y+h, x:x+w]

            face = cv2.resize(
                face,
                (IMG_SIZE, IMG_SIZE)
            )

            images.append(face)
            labels.append(current_label)

    current_label += 1

In [10]:
X = np.array(images)
y = np.array(labels)

print("Images:", X.shape)
print("Labels:", y.shape)

Images: (0,)
Labels: (0,)


In [11]:
X = X / 255.0

In [ ]:
hog_features = []

for image in X:

    feature = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )

    hog_features.append(feature)

hog_features = np.array(hog_features)

print(hog_features.shape)

In [ ]:
X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(
    hog_features,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
svm_model = SVC(
    kernel='linear',
    probability=True
)

svm_model.fit(
    X_train_ml,
    y_train_ml
)

In [ ]:
svm_predictions = svm_model.predict(X_test_ml)

print(
    classification_report(
        y_test_ml,
        svm_predictions
    )
)

In [ ]:
X_dl = np.expand_dims(X, axis=-1)

y_dl = to_categorical(y)

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_dl,
    y_dl,
    test_size=0.2,
    random_state=42
)

In [ ]:
cnn_model = Sequential([

    Conv2D(
        32,
        (3, 3),
        activation='relu',
        input_shape=(128, 128, 1)
    ),

    MaxPooling2D(2, 2),

    Conv2D(
        64,
        (3, 3),
        activation='relu'
    ),

    MaxPooling2D(2, 2),

    Conv2D(
        128,
        (3, 3),
        activation='relu'
    ),

    MaxPooling2D(2, 2),

    Flatten(),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.5),

    Dense(
        len(label_map),
        activation='softmax'
    )
])

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
cnn_model.fit(
    X_train_dl,
    y_train_dl,
    validation_data=(
        X_test_dl,
        y_test_dl
    ),
    epochs=10,
    batch_size=32
)

In [ ]:
cnn_model.save("face_recognition_model.h5")

In [ ]:
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    faces = face_cascade.detectMultiScale(
        gray,
        1.1,
        5
    )

    for (x, y, w, h) in faces:

        face = gray[y:y+h, x:x+w]

        face = cv2.resize(
            face,
            (128, 128)
        )

        normalized_face = face / 255.0

        # -------------------------
        # Machine Learning Pipeline
        # -------------------------

        hog_feature = hog(
            normalized_face,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2)
        )

        hog_feature = hog_feature.reshape(1, -1)

        svm_prediction = svm_model.predict(
            hog_feature
        )[0]

        # -------------------------
        # Deep Learning Pipeline
        # -------------------------

        cnn_input = np.expand_dims(
            normalized_face,
            axis=-1
        )

        cnn_input = np.expand_dims(
            cnn_input,
            axis=0
        )

        prediction = cnn_model.predict(
            cnn_input,
            verbose=0
        )

        confidence = np.max(prediction)

        predicted_class = np.argmax(
            prediction
        )

        name = label_map[predicted_class]

        # -------------------------
        # Відображення
        # -------------------------

        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"{name} ({confidence:.2f})",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    cv2.imshow(
        "Face Recognition Pipeline",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cnn_model = load_model(
    "face_recognition_model.h5"
)

## Datasets

In [28]:
import cv2
import os
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

dataset_path = "dataset"
os.makedirs(dataset_path, exist_ok=True)

# auto person
existing = [d for d in os.listdir(dataset_path) if d.startswith("person")]
next_id = len(existing) + 1

person_name = f"person{next_id}"
save_path = os.path.join(dataset_path, person_name)
os.makedirs(save_path, exist_ok=True)

cap = cv2.VideoCapture(0)

count = 0
max_images = 200
IMG_SIZE = 128

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    for r in results:
        boxes = r.boxes

        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            face = frame[y1:y2, x1:x2]

            if face.size == 0:
                continue

            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))

            if count < max_images:
                cv2.imwrite(f"{save_path}/{count}.jpg", face)
                count += 1

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

    cv2.putText(frame,
                f"{person_name} {count}/{max_images}",
                (10,30),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0,255,0),
                2)

    cv2.imshow("Dataset YOLO", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    if count >= max_images:
        break

cap.release()
cv2.destroyAllWindows()